In [27]:
# Import necesary libraries
from sklearn.metrics import confusion_matrix, mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from  xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [28]:
# Load the data
X_train = np.load('../data/processed/X_train_scaled.npy')
X_val   = np.load('../data/processed/X_val_scaled.npy')
X_test  = np.load('../data/processed/X_test_scaled.npy')
y_train = np.load('../data/processed/y_train.npy')
y_val   = np.load('../data/processed/y_val.npy')
y_test  = np.load('../data/processed/y_test.npy')


### SKLEARN ML PIPELINE (BASELINE MODELS)

In [29]:
### SKLEARN ML PIPELINE (BASELINE MODELS)
# Function for model evaluation
def evaluate_model(name, y_test, y_pred):
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    maxae = np.max(np.abs(y_test - y_pred))

    print(f"\n{name}")
    print(f"  R²    : {r2:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  MAPE  : {mape:.4f}")
    print(f"  MaxAE : {maxae:.4f}")


#### LINEAR REGRESSION

In [30]:
# MODEL 1:LINEAR REGRESSION
# Step 1: Define the model
lr_model = LinearRegression()
# Step 2: Train the model
lr_model.fit(X_train, y_train)
# Step 3: Predict
lr_pred = lr_model.predict(X_test)
# Evaluate the model
evaluate_model("Linear Regression", y_test, lr_pred)


Linear Regression
  R²    : 0.9128
  RMSE  : 0.0555
  MAE   : 0.0437
  MAPE  : 0.0772
  MaxAE : 0.1721


#### RANDOM FOREST REGRESSOR

In [31]:
# MODEL 2:RANDOM FOREST REGRESSOR
# Step 1: Define the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
# Step 2: Train the model
rf_model.fit(X_train, y_train)
# Step 3: Predict
rf_pred = rf_model.predict(X_test)
# Evaluate the model
evaluate_model("Random Forest Regressor", y_test, rf_pred)


Random Forest Regressor
  R²    : 0.9928
  RMSE  : 0.0159
  MAE   : 0.0090
  MAPE  : 0.0167
  MaxAE : 0.0667


#### XGBOOT REGRESSOR

In [32]:
# MODEL 3:XGBOOST REGRESSOR
# Step 1: Define the model
xgb_model = XGBRegressor(n_estimators=100, random_state=42, learning_rate = 0.1)
# Step 2: Train the model
xgb_model.fit(X_train, y_train)
# Step 3: Predict
xgb_pred = xgb_model.predict(X_test)
# Evaluate the model
evaluate_model("Xg Boost Regressor", y_test, xgb_pred)


Xg Boost Regressor
  R²    : 0.9944
  RMSE  : 0.0141
  MAE   : 0.0094
  MAPE  : 0.0175
  MaxAE : 0.0679


#### SUPPPORT VECTOR REGRESSION

In [33]:
# MODEL 4:SUPPORT VECTOR REGRESSION
# Step 1: Define the model
svr_model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.01)
# Step 2: Train the model
svr_model.fit(X_train, y_train)
# Step 3: Predict
svr_pred = svr_model.predict(X_test)
# Evaluate the model
evaluate_model("Support Vector Regressor", y_test, svr_pred)


Support Vector Regressor
  R²    : 0.9872
  RMSE  : 0.0213
  MAE   : 0.0159
  MAPE  : 0.0286
  MaxAE : 0.0691


#### MODEL COMPARISON

In [34]:
def compare_model(name, y_test, y_pred):
    return {
        'Model' : name,
        'R2'    : round(r2_score(y_test, y_pred), 4),
        'RMSE'  : round(root_mean_squared_error(y_test, y_pred), 4),
        'MAE'   : round(mean_absolute_error(y_test, y_pred), 4),
        'MAPE'  : round(mean_absolute_percentage_error(y_test, y_pred), 4),
        'MaxAE' : round(np.max(np.abs(np.array(y_test) - np.array(y_pred))), 4)
    }

results = []
results.append(compare_model("Linear Regression", y_test, lr_pred))
results.append(compare_model("SVR",               y_test, svr_pred))
results.append(compare_model("Random Forest",     y_test, rf_pred))
results.append(compare_model("XGBoost",           y_test, xgb_pred))

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

            Model     R2   RMSE    MAE   MAPE  MaxAE
          XGBoost 0.9944 0.0141 0.0094 0.0175 0.0679
    Random Forest 0.9928 0.0159 0.0090 0.0167 0.0667
              SVR 0.9872 0.0213 0.0159 0.0286 0.0691
Linear Regression 0.9128 0.0555 0.0437 0.0772 0.1721


## Baseline Model Comparison

Five models were trained and evaluated on the same test set using five
metrics: R², RMSE, MAE, MAPE, and Maximum Absolute Error (MaxAE).

| Model             | R²     | RMSE   | MAE    | MAPE   | MaxAE  |
|-------------------|--------|--------|--------|--------|--------|
| Linear Regression | 0.9128 | 0.0555 | 0.0437 | 0.0772 | 0.1721 |
| SVR               | 0.9872 | 0.0213 | 0.0159 | 0.0286 | 0.0691 |
| Random Forest     | 0.9928 | 0.0159 | 0.0090 | 0.0167 | 0.0667 |
| XGBoost           | 0.9944 | 0.0141 | 0.0094 | 0.0175 | 0.0679 |
| MLP (ANN)         | TBD    | TBD    | TBD    | TBD    | TBD    |

**Linear Regression** established the performance floor with R²=0.91.
The high MaxAE of 0.1721 confirms the flow stress relationship is
nonlinear and cannot be adequately captured by a linear model.

**SVR** showed a significant improvement by capturing nonlinearity
through the RBF kernel (R²=0.99). However it recorded the highest
MaxAE among the three strong models at 0.0691, meaning its worst
single prediction was the least reliable.

**Random Forest** performed strongly across all metrics and notably
achieved the lowest MaxAE of 0.0667 — the best worst-case prediction
of all baseline models. For an engineering application where a single
bad prediction carries real consequences, this is significant.

**XGBoost** was the strongest baseline overall, winning on R², RMSE,
and MAPE. The sequential boosting strategy — where each tree corrects
the errors of the previous one — proved well suited to this structured
experimental dataset.

**Why MLP Despite XGBoost's Performance**
XGBoost and Random Forest produce discontinuous, step-like prediction
surfaces built from if/else decision rules. Such surfaces have no
mathematical gradient, making them incompatible with gradient-based
optimization. The core objective of this project — finding optimal
input parameters — requires a smooth, differentiable surrogate model.
An ANN is built entirely from differentiable functions, making it the
only appropriate choice for the optimization stage regardless of raw
predictive performance.

### MULTI-LAYER PERCEPTRON (NEURAL NETWORK)

#### ACTIVATION FUNCTIONS
ReLU    → outputs 0 for negative inputs, x for positive
         → fast, works well for most regression problems
         → your default choice for hidden layers

Sigmoid → squashes output to (0, 1)
         → used in OUTPUT layer when target is bounded [0,1]
         → avoid in hidden layers (vanishing gradient problem)

Tanh    → squashes output to (-1, 1)
         → sometimes better than sigmoid in hidden layers
         → less common now that ReLU exists

Linear  → no transformation, output = input
         → used in output layer for unbounded regression targets

#### OPTIMIZATION
The optimizer controls how the model updates its weights during training. Think of it like the strategy for walking down a hill to find the lowest point:

SGD          → takes fixed size steps, simple but slow
Adam         → adapts step size automatically, fast and robust
              → default choice for most ANN problems
RMSprop      → good for noisy data